Cell 1 — Load features and split:

In [4]:
import numpy as np
import joblib
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

FEAT_DIR = 'D:/DDI Project/data/features/'

# mmap_mode='r' — files stay on disk, only active rows load into RAM
X = np.load(FEAT_DIR + 'X.npy', mmap_mode='r')
y = np.load(FEAT_DIR + 'y.npy', mmap_mode='r')

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

# Split by index only — no data copied yet
idx = np.arange(len(y))
idx_tr, idx_tmp = train_test_split(
    idx, test_size=0.30, random_state=42, stratify=y
)
idx_vl, idx_te = train_test_split(
    idx_tmp, test_size=0.67, random_state=42, stratify=y[idx_tmp]
)

# Save indices so 04_evaluate uses the exact same test split
np.save(FEAT_DIR + 'idx_train.npy', idx_tr)
np.save(FEAT_DIR + 'idx_val.npy',   idx_vl)
np.save(FEAT_DIR + 'idx_test.npy',  idx_te)

print(f"Train : {len(idx_tr):,}")
print(f"Val   : {len(idx_vl):,}")
print(f"Test  : {len(idx_te):,}")
print("Cell 1 done")

X shape : (133063, 8193)
y shape : (133063,)
Train : 93,144
Val   : 13,173
Test  : 26,746
Cell 1 done


Cell 2 — Load balanced training and validation samples:

In [5]:
# With 16 GB RAM we can comfortably load balanced training rows
PER_CLASS_TRAIN_MAX = 50_000   # desired cap per class
PER_CLASS_VAL_MAX   = 10_000   # desired cap per class

rng = np.random.default_rng(42)

# Separate positive and negative indices within each split
pos_tr = idx_tr[y[idx_tr] == 1]
neg_tr = idx_tr[y[idx_tr] == 0]
pos_vl = idx_vl[y[idx_vl] == 1]
neg_vl = idx_vl[y[idx_vl] == 0]

print(f"Training positives available : {len(pos_tr):,}")
print(f"Training negatives available : {len(neg_tr):,}")
print(f"Val positives available      : {len(pos_vl):,}")
print(f"Val negatives available      : {len(neg_vl):,}")

# Cap to whichever is smaller: desired size or available count
PER_CLASS_TRAIN = min(PER_CLASS_TRAIN_MAX, len(pos_tr), len(neg_tr))
PER_CLASS_VAL   = min(PER_CLASS_VAL_MAX,   len(pos_vl), len(neg_vl))
print(f"Using PER_CLASS_TRAIN = {PER_CLASS_TRAIN:,}")
print(f"Using PER_CLASS_VAL   = {PER_CLASS_VAL:,}")

# Sample balanced subsets
sample_tr = np.concatenate([
    rng.choice(pos_tr, size=PER_CLASS_TRAIN, replace=False),
    rng.choice(neg_tr, size=PER_CLASS_TRAIN, replace=False)
])
sample_vl = np.concatenate([
    rng.choice(pos_vl, size=PER_CLASS_VAL, replace=False),
    rng.choice(neg_vl, size=PER_CLASS_VAL, replace=False)
])
rng.shuffle(sample_tr)
rng.shuffle(sample_vl)

# Load into RAM as float32 — 4 bytes per value
X_tr = np.array(X[sample_tr], dtype=np.float32)
y_tr = np.array(y[sample_tr])
X_vl = np.array(X[sample_vl], dtype=np.float32)
y_vl = np.array(y[sample_vl])

print(f"\nX_train shape : {X_tr.shape}")
print(f"X_val shape   : {X_vl.shape}")
print(f"Train pos: {y_tr.sum():,}  neg: {(y_tr==0).sum():,}")
print(f"Val   pos: {y_vl.sum():,}  neg: {(y_vl==0).sum():,}")
print(f"RAM used  : ~{(X_tr.nbytes + X_vl.nbytes)/1024**3:.2f} GB")
print("Cell 2 done")

Training positives available : 46,511
Training negatives available : 46,633
Val positives available      : 6,578
Val negatives available      : 6,595
Using PER_CLASS_TRAIN = 46,511
Using PER_CLASS_VAL   = 6,578

X_train shape : (93022, 8193)
X_val shape   : (13156, 8193)
Train pos: 46,511  neg: 46,511
Val   pos: 6,578  neg: 6,578
RAM used  : ~3.24 GB
Cell 2 done



Cell 3 — Logistic Regression (baseline):

In [6]:
from sklearn.linear_model import SGDClassifier

print("Training Logistic Regression...")

lr_model = SGDClassifier(
    loss         = 'log_loss',
    max_iter     = 200,
    random_state = 42,
    n_jobs       = 1
)
lr_model.fit(X_tr, y_tr)

pred_lr = lr_model.predict(X_vl)
prob_lr = lr_model.predict_proba(X_vl)[:, 1]

print("--- Logistic Regression (Validation) ---")
print(classification_report(y_vl, pred_lr))
print(f"AUROC: {roc_auc_score(y_vl, prob_lr):.4f}")
gc.collect()

Training Logistic Regression...
--- Logistic Regression (Validation) ---
              precision    recall  f1-score   support

           0       0.67      0.71      0.69      6578
           1       0.69      0.65      0.67      6578

    accuracy                           0.68     13156
   macro avg       0.68      0.68      0.68     13156
weighted avg       0.68      0.68      0.68     13156

AUROC: 0.7502


0

Cell 4 — Random Forest:

In [7]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators = 100,
    max_depth    = 15,
    random_state = 42,
    n_jobs       = -1    # use all CPU cores
)
rf_model.fit(X_tr, y_tr)

pred_rf = rf_model.predict(X_vl)
prob_rf = rf_model.predict_proba(X_vl)[:, 1]

print("--- Random Forest (Validation) ---")
print(classification_report(y_vl, pred_rf))
print(f"AUROC: {roc_auc_score(y_vl, prob_rf):.4f}")
gc.collect()

Training Random Forest...
--- Random Forest (Validation) ---
              precision    recall  f1-score   support

           0       0.63      0.71      0.66      6578
           1       0.66      0.58      0.62      6578

    accuracy                           0.64     13156
   macro avg       0.64      0.64      0.64     13156
weighted avg       0.64      0.64      0.64     13156

AUROC: 0.6860


72

Cell 5 — XGBoost with GPU:

In [8]:
from xgboost import XGBClassifier

print("Training XGBoost (GPU)...")

xgb_model = XGBClassifier(
    n_estimators     = 500,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    tree_method      = 'hist',
    device           = 'cuda',   # RTX 4050
    eval_metric      = 'logloss',
    random_state     = 42,
    n_jobs           = -1
)
xgb_model.fit(
    X_tr, y_tr,
    eval_set  = [(X_vl, y_vl)],
    verbose   = 50
)

pred_xgb = xgb_model.predict(X_vl)
prob_xgb = xgb_model.predict_proba(X_vl)[:, 1]

print("--- XGBoost (Validation) ---")
print(classification_report(y_vl, pred_xgb))
print(f"AUROC: {roc_auc_score(y_vl, prob_xgb):.4f}")
gc.collect()

Training XGBoost (GPU)...
[0]	validation_0-logloss:0.69180
[50]	validation_0-logloss:0.66171
[100]	validation_0-logloss:0.64682
[150]	validation_0-logloss:0.63767
[200]	validation_0-logloss:0.63008
[250]	validation_0-logloss:0.62418
[300]	validation_0-logloss:0.61968
[350]	validation_0-logloss:0.61622
[400]	validation_0-logloss:0.61287
[450]	validation_0-logloss:0.61019
[499]	validation_0-logloss:0.60767


d:\DDI Project\ddienv\Lib\site-packages\xgboost\core.py:751: UserWarning: [19:54:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


--- XGBoost (Validation) ---
              precision    recall  f1-score   support

           0       0.66      0.72      0.69      6578
           1       0.69      0.63      0.66      6578

    accuracy                           0.67     13156
   macro avg       0.67      0.67      0.67     13156
weighted avg       0.67      0.67      0.67     13156

AUROC: 0.7380


0

Cell 7 — Compare all models and save the best:

In [9]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd

def get_metrics(name, y_true, y_pred, y_prob):
    return {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_true, y_pred),  4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall'   : round(recall_score(y_true, y_pred),    4),
        'F1'       : round(f1_score(y_true, y_pred),        4),
        'AUROC'    : round(roc_auc_score(y_true, y_prob),   4),
    }

results = pd.DataFrame([
    get_metrics('Logistic Regression', y_vl, pred_lr,  prob_lr),
    get_metrics('Random Forest',       y_vl, pred_rf,  prob_rf),
    get_metrics('XGBoost',             y_vl, pred_xgb, prob_xgb),
])

print("--- Validation Set Comparison ---")
print(results.to_string(index=False))
print()

# Save XGBoost as the primary model
# Best model on validation is saved
joblib.dump(xgb_model, 'D:/DDI Project/model.pkl')
print("Saved model.pkl (XGBoost)")

# Free models we are not keeping
del lr_model, rf_model
gc.collect()
print("RAM freed")
print("Next: run 04_train_neural.ipynb")

--- Validation Set Comparison ---
              Model  Accuracy  Precision  Recall     F1  AUROC
Logistic Regression    0.6836     0.6948  0.6549 0.6743 0.7502
      Random Forest    0.6424     0.6631  0.5791 0.6182 0.6860
            XGBoost    0.6720     0.6896  0.6256 0.6560 0.7380

Saved model.pkl (XGBoost)
RAM freed
Next: run 04_train_neural.ipynb
